<a href="https://colab.research.google.com/github/anokhina-rgb/Argos-Open-NMT/blob/main/pauses_text_handout_CI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# FULLY RUNNABLE GOOGLE COLAB NOTEBOOK
# Semantic sentences, sequential pauses, waveform diagram, MP3/TXT/DOCX, ZIP
# ============================================

# 1) Install dependencies
!apt-get update -qq
!apt-get install -y ffmpeg -qq
!pip install -q openai-whisper pydub matplotlib python-docx

# -------------------------
# 2) Upload audio
# -------------------------
from google.colab import files
from pydub import AudioSegment
import io

print("📂 Upload your audio file (mp3/wav):")
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

audio = AudioSegment.from_file(audio_file)
print(f"🎵 Loaded audio: {audio_file}, duration: {len(audio)/1000:.2f}s")

# -------------------------
# 3) Transcribe with Whisper
# -------------------------
import whisper

print("🔊 Transcribing audio with Whisper (base model)...")
model = whisper.load_model("base")
result = model.transcribe(audio_file)
raw_segments = result['segments']

# -------------------------
# 4) Semantic sentence merging from Whisper segments
# -------------------------
semantic_sentences = []
current_sentence = ""
start_time = None

for seg in raw_segments:
    if start_time is None:
        start_time = seg['start']
    current_sentence += " " + seg['text'].strip()
    if seg['text'].strip().endswith(('.', '!', '?')):
        semantic_sentences.append({
            "text": current_sentence.strip(),
            "start": start_time,
            "end": seg['end']
        })
        current_sentence = ""
        start_time = None

# Catch any last sentence
if current_sentence.strip():
    semantic_sentences.append({
        "text": current_sentence.strip(),
        "start": start_time,
        "end": raw_segments[-1]['end']
    })

print(f"🔀 Total semantic sentences: {len(semantic_sentences)}")

# -------------------------
# 5) Build sequential audio with pauses
# -------------------------
WAIT_MS = 300  # 0.3s wait after each sentence
output_audio = AudioSegment.silent(0)
timeline_txt = []

sentence_starts = []
sentence_ends = []
pause_starts = []
pause_ends = []
current_time = 0

for i, s in enumerate(semantic_sentences):
    start_ms = int(s['start']*1000)
    end_ms = int(s['end']*1000)
    sentence_audio = audio[start_ms:end_ms]

    seg_start = current_time
    seg_end = seg_start + len(sentence_audio)
    output_audio += sentence_audio

    timeline_txt.append(f"Sentence {i+1}: {seg_start/1000:.2f}s → {seg_end/1000:.2f}s | {s['text']}")
    sentence_starts.append(seg_start/1000)
    sentence_ends.append(seg_end/1000)

    # Add sequential pause (0.3s wait + sentence duration)
    pause_start = seg_end + WAIT_MS
    pause_end = pause_start + len(sentence_audio)
    output_audio += AudioSegment.silent(duration=WAIT_MS + len(sentence_audio))
    pause_starts.append(pause_start/1000)
    pause_ends.append(pause_end/1000)

    current_time = pause_end

# -------------------------
# 6) Export MP3 + TXT
# -------------------------
paused_mp3_file = "audio_with_pauses.mp3"
output_audio.export(paused_mp3_file, format="mp3")

paused_txt_file = "timings_with_pauses.txt"
with open(paused_txt_file, "w") as f:
    f.write("\n".join(timeline_txt))

# -------------------------
# 7) Generate waveform diagram with semantic sentences and pauses
# -------------------------
import matplotlib.pyplot as plt
import numpy as np

# Extract waveform samples
samples = np.array(audio.get_array_of_samples())
time_axis = np.linspace(0, len(audio)/1000, num=len(samples))

plt.figure(figsize=(14, max(6,len(sentence_starts)*0.5)))

# Plot waveform
plt.plot(time_axis, samples/np.max(np.abs(samples)), color='gray', alpha=0.5, label='Waveform')

# Overlay semantic sentences and pauses
for i, (s,e,p_s,p_e) in enumerate(zip(sentence_starts, sentence_ends, pause_starts, pause_ends)):
    # Highlight sentence
    plt.axvspan(s, e, color='skyblue', alpha=0.5)
    # Highlight pause
    plt.axvspan(p_s, p_e, color='lightcoral', alpha=0.3)
    # Sentence number at middle
    plt.text((s+e)/2, 0.9 - i*0.05, f"{i+1}", va='center', ha='center', fontsize=8, color='black')

plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")
plt.title("Waveform with sequential pauses (semantic sentences)")
plt.xlim(0, pause_ends[-1]+1)
plt.ylim(-1.1, 1.1)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()

diagram_file="waveform_semantic.png"
plt.savefig(diagram_file)
plt.close()
print(f"✅ Waveform diagram saved as {diagram_file}")

# -------------------------
# 8) Create DOCX handout
# -------------------------
from docx import Document
from docx.shared import Inches

doc = Document()
doc.add_heading("Student Handout: Audio Segments", level=1)

# Page 1 - Original transcript
doc.add_heading("Page 1: Original Transcript", level=2)
doc.add_paragraph(" ".join([s['text'] for s in semantic_sentences]))
doc.add_picture(diagram_file, width=Inches(6))
doc.add_page_break()

# Page 2 - Transcript with sequential pauses
doc.add_heading("Page 2: Transcript with Sequential Pauses", level=2)
doc.add_paragraph("\n".join(timeline_txt))
doc.add_picture(diagram_file, width=Inches(6))

docx_file="handout_with_pauses.docx"
doc.save(docx_file)

# -------------------------
# 9) Package everything into ZIP
# -------------------------
import zipfile
from google.colab import files

zip_filename="handout_package.zip"
with zipfile.ZipFile(zip_filename,'w') as zipf:
    zipf.write(paused_mp3_file)
    zipf.write(paused_txt_file)
    zipf.write(diagram_file)
    zipf.write(docx_file)

files.download(zip_filename)
print("✅ All files packaged in ZIP and ready for download.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.2 MB/s eta 0:00:00
